# ĐÁNH GIÁ

In [1]:
import polars as pl
from pathlib import Path
import gc
import shutil

# --- 1. KHAI BÁO ĐƯỜNG DẪN THƯ MỤC VÀ FILE ---
thu_muc_chinh = Path(r"C:\Users\lamnv5_vtp\Downloads\DanhGia")
thu_muc_tam = thu_muc_chinh / "temp_parquet"
thu_muc_tam.mkdir(exist_ok=True)

# Đường dẫn file Parquet gốc tích lũy dữ liệu
file_parquet_goc = thu_muc_chinh / "Data_FM_MM_LM.parquet"
file_tam_goc = thu_muc_chinh / "Data_FM_MM_LM_temp_master.parquet"

# Quét tìm tất cả các file Excel mới có dạng "Thời gian phân đoạn bưu gửi_..."
excel_files = list(thu_muc_chinh.glob("Thời gian phân đoạn bưu gửi_*.xls*"))

if len(excel_files) > 0:
    print(f"📁 Tìm thấy {len(excel_files)} file Excel mới. Đang tiến hành xử lý...")

    count = 0
    for file_path in excel_files:
        try:
            # Đọc file Excel bằng engine calamine (giúp tối ưu tốc độ và tránh tràn RAM)
            df = pl.read_excel(file_path, engine="calamine", infer_schema_length=0)
            
            # Gắn thêm cột tên nguồn file để dễ truy xuất vết nếu cần
            df = df.with_columns(pl.lit(file_path.name).alias("file_source"))

            # Ghi ra file parquet tạm thời
            out_temp = thu_muc_tam / f"part_{count}.parquet"
            df.write_parquet(out_temp)
            count += 1

            # Ép giải phóng bộ nhớ RAM ngay lập tức cho từng vòng lặp
            del df
            gc.collect()
            print(f"  [{count}/{len(excel_files)}] Đã chuyển đổi xong: {file_path.name}")

        except Exception as e:
            print(f"❌ Lỗi khi đọc file {file_path.name}: {e}")

    if count > 0:
        print("\n🔄 Đang tiến hành gộp dữ liệu mới vào file Parquet gốc (chạy chế độ Streaming)...")

        # Gom nguồn quét: Bao gồm file parquet gốc cũ (nếu đã có sẵn) + toàn bộ file tạm mới
        danh_sach_scan = [str(thu_muc_tam / "*.parquet")]
        if file_parquet_goc.exists():
            danh_sach_scan.append(str(file_parquet_goc))

        # Nếu file tạm gộp trước đó còn sót lại thì xóa trước cho sạch sẽ
        if file_tam_goc.exists():
            file_tam_goc.unlink()

        # Quét và xả dữ liệu trực tiếp xuống file tạm master bằng streaming
        pl.scan_parquet(danh_sach_scan).sink_parquet(file_tam_goc)

        # Thay thế file gốc cũ bằng file gốc mới hoàn chỉnh
        if file_parquet_goc.exists():
            file_parquet_goc.unlink()
        file_tam_goc.rename(file_parquet_goc)

        print(f"✅ Đã cập nhật thành công vào file Parquet gốc: {file_parquet_goc}")

        # --- 2. DỌN DẸP TOÀN BỘ THEO YÊU CẦU ---
        print("\n🧹 Đang dọn dẹp các tệp dữ liệu thừa...")

        # Xóa thư mục chứa các file parquet phần nhỏ tạm thời
        shutil.rmtree(thu_muc_tam)
        print("  - Đã xóa thư mục parquet tạm.")

        # Xóa luôn các file Excel gốc vừa xử lý thành công
        for file_path in excel_files:
            try:
                file_path.unlink()
                print(f"  - Đã xóa file Excel cũ: {file_path.name}")
            except Exception as e:
                print(f"⚠️ Không thể xóa file {file_path.name}: {e}")

        print("\n✨ HOÀN TẤT! Thư mục đã được làm sạch, dữ liệu mới đã được gộp gọn gàng vào file Parquet gốc.")
    else:
        print("⚠️ Không trích xuất được dữ liệu hợp lệ nào từ các file Excel.")
        if thu_muc_tam.exists():
            shutil.rmtree(thu_muc_tam)
else:
    print("📭 Không tìm thấy file Excel nào mới trong thư mục để xử lý!")

📁 Tìm thấy 3 file Excel mới. Đang tiến hành xử lý...
  [1/3] Đã chuyển đổi xong: Thời gian phân đoạn bưu gửi_2026_09_21__1.xlsx
  [2/3] Đã chuyển đổi xong: Thời gian phân đoạn bưu gửi_2026_09_21__2.xlsx
  [3/3] Đã chuyển đổi xong: Thời gian phân đoạn bưu gửi_2026_09_21__3.xlsx

🔄 Đang tiến hành gộp dữ liệu mới vào file Parquet gốc (chạy chế độ Streaming)...
✅ Đã cập nhật thành công vào file Parquet gốc: C:\Users\lamnv5_vtp\Downloads\DanhGia\Data_FM_MM_LM.parquet

🧹 Đang dọn dẹp các tệp dữ liệu thừa...
  - Đã xóa thư mục parquet tạm.
  - Đã xóa file Excel cũ: Thời gian phân đoạn bưu gửi_2026_09_21__1.xlsx
  - Đã xóa file Excel cũ: Thời gian phân đoạn bưu gửi_2026_09_21__2.xlsx
  - Đã xóa file Excel cũ: Thời gian phân đoạn bưu gửi_2026_09_21__3.xlsx

✨ HOÀN TẤT! Thư mục đã được làm sạch, dữ liệu mới đã được gộp gọn gàng vào file Parquet gốc.
